In [2]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor, load_bg
from recOrder.recOrder.io.reader import MicromanagerReader
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

In [3]:
def add_colorbar(mappable):
    
    last_axes = plt.gca()
    ax = mappable.axes
    fig = ax.figure
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size='5%', pad=0.05)
    cbar = fig.colorbar(mappable, cax=cax)
    plt.sca(last_axes)
    return cbar

In [4]:
def init_zarr_store(data_path, chunk_size=(1, 65, 2048, 2048), data_shape = (10,65,2048,2048)):
    compressor=Blosc(cname='zstd', clevel=1, shuffle=Blosc.SHUFFLE)
    
    if os.path.exists(data_path):
        raise ValueError('Zarr Store already exists, please open the existing store to avoid overwriting data')
    
    if not data_path.endswith('.zarr'):
        data_path += '.zarr'
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
        
    
    zarr_store = zarr.open(data_path, mode='a')
    
    
    datasets = ['Phase3D', 'Retardance', 'Orientation', 'BF']
#     groups = ['RSV48', 'RSV24', 'Mock48', 'Mock24']
    
#     for group in groups:
#         zarr_store.create_group(group)
#         for ds in datasets:
#             zarr_store[group].zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
#                                         compressor=compressor)
    for ds in datasets:
        zarr_store.zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
                                        compressor=compressor)
        
    ## ADD TO INITIALIZE ADDITIONAL FLUOR ARRAYS - COPY PASTE
    zarr_store.zeros('GFP', shape=data_shape, chunks=chunk_size, dtype='uint16',
                                        compressor=compressor)

    
    return zarr_store

## Gather Data Paths

In [23]:
folder = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/RSV_IFNL_24_1_2/_1'
data = MicromanagerReader(folder, extract_data=True)

[INFO: reader: 131 2021-03-19 14:50:35,723] checking _1_MMStack_7.ome.tif for ome-master records
[WARNING: reader: 154 2021-03-19 14:50:36,009] OME series: Master-ome found
[WARNING: tifffile:15794 2021-03-19 14:50:36,206] read_json: invalid JSON
[INFO: reader: 105 2021-03-19 14:50:36,207] extracting data from /gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/RSV_IFNL_24_1_2/_1/_1_MMStack.ome.tif


## Initialize Data Saving

In [24]:
data_save_folder = '/gpfs/CompMicro/projects/A549/2021_02_25_40X_04NA_A549'
condition = 'RSV_IFNL_24'

chunk_size = (1, 65, 2048, 2048)
data_shape = (10,65,2048,2048)


zarr_store = init_zarr_store(os.path.join(data_save_folder,condition), chunk_size=chunk_size, data_shape=data_shape)
zarr_store.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, icon='table', name='BF (10, 65, 2048, 204…

## Initialize Reconstruction

In [20]:
## Set Reconstruction Parameters
image_dim = (2048,2048)
wavelength = 525
swing = 0.03
N_channel = 4
NA_obj = 1.1
NA_illu = 0.4
mag = 40
N_slices = 65
z_step = 0.25
pad_z = 0
pixel_size = 6.5
bg_option = 'local_fit'
n_media = 1.33

reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)



Initializing Reconstructor...
Finished Initializing Reconstructor (1.64 min)


## Reconstruct Batch

In [21]:
bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_25_40X_04NA_A549/Calibration/BG/'
bg_data = load_bg(bg_path, 2048, 2048, ROI=(0,500,1000,250), cropped=True)

In [25]:
for pos in range(data.get_num_positions()):
    ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(data._positions[pos], bg_data, reconstructor, method = "Tikhonov",
                                                                   reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                   lambda_re = 1e-3, lambda_im = 1e-3, itr = 20)
        
    zarr_store['BF'][pos] = BF_stack
    zarr_store['Retardance'][pos] = ret_stack
    zarr_store['Orientation'][pos] = ori_stack
    zarr_store['Phase3D'][pos] = np.transpose(phase3D,(2,0,1))
    
    # SAVE ADDITIONAL FLUOR CHANNELS HERE, INDEX INTO CORRECT CHANNEL
    zarr_store['GFP'][pos] = data._positions[pos][-1]
    

Computing Birefringence...
Finished Computing Birefringence (1.58 min)
Computing 3d Phase...
Finished Reconstruction (2.18 min)

Computing Birefringence...
Finished Computing Birefringence (1.56 min)
Computing 3d Phase...
Finished Reconstruction (2.12 min)

Computing Birefringence...
Finished Computing Birefringence (1.57 min)
Computing 3d Phase...
Finished Reconstruction (2.14 min)

Computing Birefringence...
Finished Computing Birefringence (1.55 min)
Computing 3d Phase...
Finished Reconstruction (2.12 min)

Computing Birefringence...
Finished Computing Birefringence (1.56 min)
Computing 3d Phase...
Finished Reconstruction (2.13 min)

Computing Birefringence...
Finished Computing Birefringence (1.55 min)
Computing 3d Phase...
Finished Reconstruction (2.12 min)

Computing Birefringence...
Finished Computing Birefringence (1.56 min)
Computing 3d Phase...
Finished Reconstruction (2.16 min)

Computing Birefringence...
Finished Computing Birefringence (1.57 min)
Computing 3d Phase...
Fini